# Olist E-Commerce — EDA
**Day 2 Notebook** | Four business questions answered with charts and written findings.

Load `master_orders.csv` once at the top, then work from filtered subsets throughout.

## 1. Imports & Load Data

In [49]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)


In [50]:

master = pd.read_csv(
    '../processed/master_orders.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ]
)
  
print(f'Loaded: {master.shape[0]:,} rows x {master.shape[1]} cols')
print(f'Date range: {master["order_purchase_timestamp"].min().date()} to {master["order_purchase_timestamp"].max().date()}')

Loaded: 99,992 rows x 27 cols
Date range: 2016-09-04 to 2018-10-17


## 2. Create Filtered Subsets
Work from clean subsets throughout — never plot raw `master` directly.
This prevents cancelled/pending orders from polluting your charts.

In [51]:
# delivered orders only
delivered = master[master['order_status'] == 'delivered'].copy()

# delivered + has review score
reviewed = delivered[delivered['review_score'].notna()].copy()

# delivered + has order value
revenue_df = delivered[delivered['order_value'].notna()].copy()

print(f'master:     {len(master):,} orders')
print(f'delivered:  {len(delivered):,} orders  ({len(delivered)/len(master)*100:.1f}% of total)')
print(f'reviewed:   {len(reviewed):,} orders  ({len(reviewed)/len(delivered)*100:.1f}% of delivered)')
print(f'revenue_df: {len(revenue_df):,} orders')

master:     99,992 orders
delivered:  97,007 orders  (97.0% of total)
reviewed:   96,361 orders  (99.3% of delivered)
revenue_df: 97,007 orders


---
## Section 1 — Revenue & Growth
*Question: How did Olist's business perform over time, and which categories drive it?*

### 1a. Monthly GMV

In [52]:
monthly = (
    revenue_df
    .set_index('order_purchase_timestamp')
    .resample('M')['order_value']
    .agg(['sum', 'count', 'mean'])
    .reset_index()
    .rename(columns={'sum': 'gmv', 'count': 'n_orders', 'mean': 'avg_order_value'})
)

# remove first and last month (partial months — misleading)
monthly = monthly.iloc[1:-1].copy()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Monthly GMV (R$)', 'Monthly Order Count'],
    vertical_spacing=0.12
)

fig.add_trace(
    go.Bar(x=monthly['order_purchase_timestamp'], y=monthly['gmv'],
           marker_color='#4ade80', name='GMV'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=monthly['order_purchase_timestamp'], y=monthly['n_orders'],
               mode='lines+markers', line=dict(color='#60a5fa'), name='Orders'),
    row=2, col=1
)

fig.update_layout(
    height=500, showlegend=False,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0'),
    title='Olist Monthly Performance'
)
fig.show()

# key numbers
peak = monthly.loc[monthly['gmv'].idxmax()]
print(f'Peak GMV month: {peak["order_purchase_timestamp"].strftime("%b %Y")} — R$ {peak["gmv"]:,.0f}')
print(f'Total GMV: R$ {monthly["gmv"].sum():,.0f}')
print(f'Growth (first vs last month): {(monthly["gmv"].iloc[-1] / monthly["gmv"].iloc[0] - 1)*100:.0f}%')

C:\Users\hoaphung\AppData\Local\Temp\ipykernel_15932\251242112.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M')['order_value']


Peak GMV month: Nov 2017 — R$ 995,200
Total GMV: R$ 12,441,051
Growth (first vs last month): 2050%


**Finding:** *(fill in after running)*
e.g. GMV grew from R$X in Jan 2017 to R$Y by Aug 2018. A clear spike is visible in Nov 2017 (Black Friday). Growth plateaued from Q2 2018.

### 1b. Revenue by Product Category

In [53]:
cat_revenue = (
    revenue_df
    .groupby('category_english')['order_value']
    .agg(['sum', 'count', 'mean'])
    .reset_index()
    .rename(columns={'sum': 'total_gmv', 'count': 'n_orders', 'mean': 'avg_order_value'})
    .sort_values('total_gmv', ascending=False)
    .head(15)
)

fig = px.bar(
    cat_revenue, x='total_gmv', y='category_english',
    orientation='h',
    color='avg_order_value',
    color_continuous_scale='Greens',
    labels={'total_gmv': 'Total GMV (R$)', 'category_english': '',
            'avg_order_value': 'Avg Order Value'},
    title='Top 15 Categories by Total Revenue'
)
fig.update_layout(
    height=480, yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

print('Top 5 categories by GMV:')
print(cat_revenue[['category_english','total_gmv','n_orders','avg_order_value']].head(5).to_string(index=False))

Top 5 categories by GMV:
     category_english    total_gmv  n_orders  avg_order_value
        health_beauty 1,236,382.35      8661           142.75
        watches_gifts 1,164,729.90      5482           212.46
       bed_bath_table 1,035,894.05      9282           111.60
       sports_leisure   961,434.42      7541           127.49
computers_accessories   894,699.82      6551           136.57


**Finding:** *(fill in after running)*
e.g. Bed/bath/table and health/beauty are the top two categories. Computers have fewer orders but highest avg order value.

### 1c. Weekday & Hour Patterns

In [54]:
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

by_weekday = (
    revenue_df.groupby('purchase_weekday')['order_id']
    .count().reset_index()
    .rename(columns={'order_id': 'n_orders'})
)
by_weekday['day_name'] = by_weekday['purchase_weekday'].map(dict(enumerate(day_names)))

by_hour = (
    revenue_df.groupby('purchase_hour')['order_id']
    .count().reset_index()
    .rename(columns={'order_id': 'n_orders'})
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Orders by Weekday', 'Orders by Hour of Day'])

fig.add_trace(
    go.Bar(x=by_weekday['day_name'], y=by_weekday['n_orders'],
           marker_color='#4ade80', name='Weekday'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=by_hour['purchase_hour'], y=by_hour['n_orders'],
           marker_color='#60a5fa', name='Hour'),
    row=1, col=2
)

fig.update_layout(
    height=380, showlegend=False,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0'),
    title='When Do Customers Order?'
)
fig.show()

**Finding:** *(fill in after running)*
e.g. Orders peak Tuesday-Wednesday. The 8pm hour is the busiest — customers shop after work.

---
## Section 2 — Delivery Performance
*Question: Does delivery quality affect customer satisfaction?*
This is the strongest statistical story in the dataset.

### 2a. Delivery Delay Distribution

In [55]:
# work only with delivered orders that have delay calculated
delay_df = reviewed[reviewed['delivery_delay_days'].notna()].copy()

print(f'Orders delivered on time or early: {(delay_df["delivery_delay_days"] <= 0).sum():,} '
      f'({(delay_df["delivery_delay_days"] <= 0).mean()*100:.1f}%)')
print(f'Orders delivered late:             {(delay_df["delivery_delay_days"] > 0).sum():,} '
      f'({(delay_df["delivery_delay_days"] > 0).mean()*100:.1f}%)')
print(f'Avg delay (late orders only):      {delay_df[delay_df["delivery_delay_days"]>0]["delivery_delay_days"].mean():.1f} days')

fig = px.histogram(
    delay_df,
    x='delivery_delay_days',
    nbins=80,
    color_discrete_sequence=['#4ade80'],
    labels={'delivery_delay_days': 'Delivery Delay (days, positive=late)'},
    title='Distribution of Delivery Delay'
)
fig.add_vline(x=0, line_dash='dash', line_color='#fb923c',
              annotation_text='On time', annotation_position='top right')
fig.update_layout(
    height=380,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

Orders delivered on time or early: 89,944 (93.3%)
Orders delivered late:             6,409 (6.7%)
Avg delay (late orders only):      10.5 days


### 2b. Review Score vs Delivery Delay

In [56]:
# bucket delay into groups
bins   = [-999, -7, 0, 3, 7, 14, 999]
labels = ['Early >7d', 'On time / Early', '1-3 days late',
           '4-7 days late', '8-14 days late', '>14 days late']

delay_df['delay_bucket'] = pd.cut(
    delay_df['delivery_delay_days'], bins=bins, labels=labels
)

bucket_stats = (
    delay_df.groupby('delay_bucket', observed=True)['review_score']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_review', 'count': 'n_orders'})
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Avg Review Score by Delay Bucket', 'Order Count per Bucket']
)

colors = ['#4ade80','#4ade80','#fbbf24','#fb923c','#ef4444','#991b1b']

fig.add_trace(
    go.Bar(x=bucket_stats['delay_bucket'].astype(str),
           y=bucket_stats['avg_review'],
           marker_color=colors, name='Avg Review'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=bucket_stats['delay_bucket'].astype(str),
           y=bucket_stats['n_orders'],
           marker_color='#60a5fa', name='Count'),
    row=1, col=2
)

fig.update_layout(
    height=400, showlegend=False,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0'),
    title='How Delivery Delay Affects Review Score'
)
fig.show()

print('\nReview score by delay bucket:')
print(bucket_stats.to_string(index=False))


Review score by delay bucket:
   delay_bucket  avg_review  n_orders
      Early >7d        4.31     76175
On time / Early        4.16     13769
  1-3 days late        3.29      1856
  4-7 days late        2.10      1756
 8-14 days late        1.68      1452
  >14 days late        1.72      1345


**Finding:** *(fill in after running)*
e.g. On-time orders average 4.3 stars. Orders >14 days late average X stars. The drop is sharp after the 7-day threshold.

### 2c. Correlation — Delay vs Review Score

In [57]:
from scipy import stats

clean = delay_df[['delivery_delay_days','review_score']].dropna()
corr, pval = stats.spearmanr(clean['delivery_delay_days'], clean['review_score'])
print(f'Spearman correlation (delay vs review score): {corr:.3f}  (p={pval:.2e})')
print('Interpretation: negative = more delay → lower score')

# 1-star review rate by delay bucket
delay_df['is_1star'] = delay_df['review_score'] == 1
one_star_rate = (
    delay_df.groupby('delay_bucket', observed=True)['is_1star']
    .mean().mul(100).round(1)
    .reset_index()
    .rename(columns={'is_1star': '1star_rate_%'})
)
print('\n1-star review rate by delay bucket:')
print(one_star_rate.to_string(index=False))

Spearman correlation (delay vs review score): -0.176  (p=0.00e+00)
Interpretation: negative = more delay → lower score

1-star review rate by delay bucket:
   delay_bucket  1star_rate_%
      Early >7d          6.50
On time / Early          7.40
  1-3 days late         25.20
  4-7 days late         58.50
 8-14 days late         70.40
  >14 days late         68.90


**Finding:** *(fill in after running)*
e.g. Spearman r = -0.XX. 1-star review rate jumps from X% for on-time orders to Y% for orders >14 days late.

### 2d. Delivery Duration by State

In [58]:
state_delivery = (
    delivered[delivered['delivery_days'].notna()]
    .groupby('customer_state')['delivery_days']
    .agg(['mean', 'median', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_days', 'median': 'median_days', 'count': 'n_orders'})
    .sort_values('avg_days', ascending=False)
)

fig = px.bar(
    state_delivery, x='customer_state', y='avg_days',
    color='avg_days',
    color_continuous_scale='RdYlGn_r',
    labels={'customer_state': 'State', 'avg_days': 'Avg Delivery Days'},
    title='Average Delivery Duration by Brazilian State'
)
fig.update_layout(
    height=400,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

print('Fastest states (avg delivery days):')
print(state_delivery.tail(5)[['customer_state','avg_days','n_orders']].to_string(index=False))
print('\nSlowest states:')
print(state_delivery.head(5)[['customer_state','avg_days','n_orders']].to_string(index=False))

Fastest states (avg delivery days):
customer_state  avg_days  n_orders
            SC     14.50      3560
            DF     12.54      2099
            MG     11.55     11423
            PR     11.54      4942
            SP      8.30     40705

Slowest states:
customer_state  avg_days  n_orders
            RR     28.98        41
            AP     26.73        67
            AM     25.95       146
            AL     24.03       401
            PA     23.32       952


**Finding:** *(fill in after running)*
e.g. SP and RJ average ~X days. Northern states like RR, AP, AM average ~Y days — 2-3x slower.

---
## Section 3 — Customer Geography
*Question: Where are Olist's customers and what does that mean for the business?*

### 3a. Order Volume by State

In [59]:
state_orders = (
    delivered
    .groupby('customer_state')
    .agg(
        n_orders      = ('order_id',      'count'),
        total_gmv     = ('order_value',   'sum'),
        avg_order_val = ('order_value',   'mean'),
        avg_freight   = ('freight_value', 'mean'),
    )
    .reset_index()
)

state_orders['freight_ratio'] = (
    state_orders['avg_freight'] / state_orders['avg_order_val']
).round(3)

state_orders['order_share_%'] = (
    state_orders['n_orders'] / state_orders['n_orders'].sum() * 100
).round(1)

fig = px.bar(
    state_orders.sort_values('n_orders', ascending=False),
    x='customer_state', y='n_orders',
    color='avg_order_val',
    color_continuous_scale='Blues',
    labels={'customer_state': 'State', 'n_orders': 'Order Count',
            'avg_order_val': 'Avg Order Value (R$)'},
    title='Order Volume by State (colour = avg order value)'
)
fig.update_layout(
    height=400,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

sp_share = state_orders.loc[state_orders['customer_state']=='SP', 'order_share_%'].values[0]
print(f'SP share of all orders: {sp_share:.1f}%')
top3 = state_orders.nlargest(3,'n_orders')[['customer_state','n_orders','order_share_%']]
print(f'Top 3 states: {top3.to_string(index=False)}')

SP share of all orders: 42.0%
Top 3 states: customer_state  n_orders  order_share_%
            SP     40712          42.00
            RJ     12420          12.80
            MG     11423          11.80


### 3b. Freight Burden by State

In [60]:
fig = px.bar(
    state_orders.sort_values('freight_ratio', ascending=False),
    x='customer_state', y='freight_ratio',
    color='freight_ratio',
    color_continuous_scale='Oranges',
    labels={'customer_state': 'State',
            'freight_ratio': 'Freight as % of Order Value'},
    title='Freight Cost Burden by State (freight / order value)'
)
fig.update_layout(
    height=400,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

print('Highest freight burden states (freight as % of order value):')
print(state_orders.nlargest(5,'freight_ratio')[['customer_state','freight_ratio','avg_freight','avg_order_val']].to_string(index=False))
print('\nLowest freight burden states:')
print(state_orders.nsmallest(5,'freight_ratio')[['customer_state','freight_ratio','avg_freight','avg_order_val']].to_string(index=False))

Highest freight burden states (freight as % of order value):
customer_state  freight_ratio  avg_freight  avg_order_val
            RR           0.28        48.34         172.13
            MA           0.26        42.91         162.56
            RO           0.25        46.43         187.99
            AM           0.25        37.38         152.14
            PI           0.24        42.93         177.74

Lowest freight burden states:
customer_state  freight_ratio  avg_freight  avg_order_val
            SP           0.14        17.34         125.11
            MS           0.16        26.93         164.09
            DF           0.17        23.87         142.21
            RJ           0.17        23.94         142.21
            MG           0.17        23.45         136.45


**Finding:** *(fill in after running)*
e.g. Northern states (RR, AP, AM) have freight ratios of X% vs SP at Y% — nearly 3x higher. Customers in these regions pay proportionally much more to receive the same goods.

---
## Section 4 — Seller Quality
*Question: Are all sellers equal, and do bad sellers hurt the platform disproportionately?*

### 4a. Seller-level Metrics

In [61]:
seller_stats = (
    reviewed
    .groupby('seller_id')
    .agg(
        n_orders       = ('order_id',      'count'),
        avg_review     = ('review_score',  'mean'),
        total_revenue  = ('order_value',   'sum'),
        avg_delay      = ('delivery_delay_days', 'mean'),
    )
    .reset_index()
)

seller_stats['is_1star_heavy'] = (
    reviewed.groupby('seller_id')
    .apply(lambda x: (x['review_score']==1).mean())
    .values
)

print(f'Total sellers: {len(seller_stats):,}')
print(f'Avg review score across sellers: {seller_stats["avg_review"].mean():.2f}')
print(f'Sellers with avg review < 3.0: {(seller_stats["avg_review"] < 3.0).sum():,} '
      f'({(seller_stats["avg_review"] < 3.0).mean()*100:.1f}%)')

Total sellers: 2,956
Avg review score across sellers: 4.20
Sellers with avg review < 3.0: 147 (5.0%)


C:\Users\hoaphung\AppData\Local\Temp\ipykernel_15932\2872409441.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['review_score']==1).mean())


### 4b. Review Score Distribution Across Sellers

In [62]:
fig = px.histogram(
    seller_stats[seller_stats['n_orders'] >= 10],  # sellers with 10+ orders only
    x='avg_review',
    nbins=40,
    color_discrete_sequence=['#c084fc'],
    labels={'avg_review': 'Seller Average Review Score'},
    title='Distribution of Seller Review Scores (sellers with 10+ orders)'
)
fig.update_layout(
    height=380,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

### 4c. Volume vs Quality Scatter

In [63]:
active_sellers = seller_stats[seller_stats['n_orders'] >= 20].copy()

fig = px.scatter(
    active_sellers,
    x='n_orders', y='avg_review',
    size='total_revenue',
    color='avg_delay',
    color_continuous_scale='RdYlGn_r',
    hover_data=['seller_id','n_orders','avg_review','total_revenue'],
    labels={
        'n_orders': 'Number of Orders',
        'avg_review': 'Avg Review Score',
        'avg_delay': 'Avg Delivery Delay (days)'
    },
    title='Seller Volume vs Quality (size=revenue, colour=avg delay)'
)
fig.update_layout(
    height=480,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

### 4d. Impact of Bottom Sellers

In [64]:
# bottom 5% of sellers by avg review score
threshold = seller_stats['avg_review'].quantile(0.05)
bad_sellers = seller_stats[seller_stats['avg_review'] <= threshold]['seller_id']

bad_orders = reviewed[reviewed['seller_id'].isin(bad_sellers)]
total_1star = (reviewed['review_score'] == 1).sum()
bad_1star   = (bad_orders['review_score'] == 1).sum()

print(f'Bottom 5% sellers:          {len(bad_sellers):,} sellers')
print(f'Their share of all orders:  {len(bad_orders)/len(reviewed)*100:.1f}%')
print(f'Their share of 1-star reviews: {bad_1star/total_1star*100:.1f}%')
print(f'\n=> Bottom 5% of sellers generate {bad_1star/total_1star*100:.0f}% of all 1-star reviews')

Bottom 5% sellers:          233 sellers
Their share of all orders:  0.9%
Their share of 1-star reviews: 4.2%

=> Bottom 5% of sellers generate 4% of all 1-star reviews


**Finding:** *(fill in after running)*
e.g. Bottom 5% of sellers account for X% of orders but Y% of all 1-star reviews — a disproportionate quality drag on the platform.